In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import joblib
import json
import os
from pathlib import Path

In [31]:
# Project paths
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")
ARTIFACTS_DIR = Path("../models/preprocessing")

# Create output folders if they do not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TRANSACTION_PATH = RAW_DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = RAW_DATA_DIR / "train_identity.csv"

TARGET = "isFraud"
ID_COL = "TransactionID"
TIME_COL = "TransactionDT"

In [32]:
# Load datasets
train_transactions = pd.read_csv(TRAIN_TRANSACTION_PATH)
train_identity = pd.read_csv(TRAIN_IDENTITY_PATH)

print(f"Train transactions shape: {train_transactions.shape}")
print(f"Train identity shape: {train_identity.shape}")

Train transactions shape: (590540, 394)
Train identity shape: (144233, 41)


In [33]:
train_transactions.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
df = train_transactions.merge(train_identity, on = ID_COL, how = 'left')
print(f"Merged dataset shape: {df.shape}")
df.head()

Merged dataset shape: (590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [35]:
fraud_counts = df[TARGET].value_counts()
fraud_rate = df[TARGET].mean() * 100
print(f"Fraud counts:\n{fraud_counts}")
print(f"Fraud rate: {fraud_rate:.4f}%")

Fraud counts:
isFraud
0    569877
1     20663
Name: count, dtype: int64
Fraud rate: 3.4990%


In [37]:
# Memory Usage
print("Memory usage:")
print(df.memory_usage(deep = True).sum() / 1e6, "MB")

Memory usage:
2691.793804 MB


In [38]:
# Sort by transaction time
df = df.sort_values(by = TIME_COL).reset_index(drop = True)

print(df[[ID_COL, TIME_COL, TARGET]].head())
print(df[[ID_COL, TIME_COL, TARGET]].tail())

   TransactionID  TransactionDT  isFraud
0        2987000          86400        0
1        2987001          86401        0
2        2987002          86469        0
3        2987003          86499        0
4        2987004          86506        0
        TransactionID  TransactionDT  isFraud
590535        3577535       15811047        0
590536        3577536       15811049        0
590537        3577537       15811079        0
590538        3577538       15811088        0
590539        3577539       15811131        0


In [39]:
# Split into train and validation sets based on time

n = len(df)

train_df = df.iloc[ : int(n * 0.6)]
val_df = df.iloc[int(n * 0.6) : int(n * 0.8)]
prod_df = df.iloc[int(n * 0.8) : ]

In [41]:
train_df.shape, val_df.shape, prod_df.shape

((354324, 434), (118108, 434), (118108, 434))

In [42]:
# Seperate features and target
X_train = train_df.drop(columns = [TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns = [TARGET])
y_val = val_df[TARGET]

X_prod = prod_df.drop(columns = [TARGET])
y_prod = prod_df[TARGET]

In [43]:
# Save ID and Time columns for later use in monitoring and production
train_metadata = X_train[[ID_COL, TIME_COL]].copy()
val_metadata = X_val[[ID_COL, TIME_COL]].copy()
prod_metadata = X_prod[[ID_COL, TIME_COL]].copy()

In [44]:
# Drop ID_COL and TIME_COL from features since they won't be used for modeling
DROP_COLS = [ID_COL, TIME_COL]

X_train = X_train.drop(columns = DROP_COLS)
X_val = X_val.drop(columns = DROP_COLS)
X_prod = X_prod.drop(columns = DROP_COLS)

In [45]:
# Store categorical and numerical column names for later use in modeling and monitoring
categorical_cols = X_train.select_dtypes(include = ["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include = ["number"]).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

Categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
Numerical columns: ['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59

In [46]:
# Fill categorical missing values with "missing"
X_train[categorical_cols] = X_train[categorical_cols].astype("string").fillna("missing")
X_val[categorical_cols] = X_val[categorical_cols].astype("string").fillna("missing")
X_prod[categorical_cols] = X_prod[categorical_cols].astype("string").fillna("missing")

In [47]:
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    
    # Fit only on training categories to avoid future leakage
    le.fit(X_train[col])
    
    encoders[col] = le
    
    # Helper function to handle unseen categories in validation/production
    known_classes = set(le.classes_)
    
    def encode_with_unknown(series, label_encoder, known_classes):
        series = series.astype("string").fillna("missing")
        series = series.apply(lambda x: x if x in known_classes else "unknown")
        
        # If "unknown" was not in training classes, add it manually
        if "unknown" not in label_encoder.classes_:
            label_encoder.classes_ = np.append(label_encoder.classes_, "unknown")
            
        return label_encoder.transform(series)
    
    X_train[col] = le.transform(X_train[col])
    X_val[col] = encode_with_unknown(X_val[col], le, known_classes)
    X_prod[col] = encode_with_unknown(X_prod[col], le, known_classes)

print("Categorical encoding complete.")

Categorical encoding complete.


In [48]:
X_train.head()

,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,68.5,4,13926,NaN,150.0,1,142.0,1,315.0,87.0,...,76,NaN,163,4,2,2,2,2,1,1405
1,29.0,4,2755,404.0,150.0,2,102.0,1,325.0,87.0,...,76,NaN,163,4,2,2,2,2,1,1405
2,59.0,4,4663,490.0,150.0,4,166.0,2,330.0,87.0,...,76,NaN,163,4,2,2,2,2,1,1405
3,50.0,4,18132,567.0,150.0,2,117.0,2,476.0,87.0,...,76,NaN,163,4,2,2,2,2,1,1405
4,50.0,1,4497,514.0,150.0,2,102.0,1,420.0,87.0,...,97,32.0,97,3,1,0,1,1,2,767


In [50]:
NUMERIC_MISSING_VALUE = -999

X_train[numerical_cols] = X_train[numerical_cols].fillna(NUMERIC_MISSING_VALUE)
X_val[numerical_cols] = X_val[numerical_cols].fillna(NUMERIC_MISSING_VALUE)
X_prod[numerical_cols] = X_prod[numerical_cols].fillna(NUMERIC_MISSING_VALUE)

In [51]:
processed_train = X_train.copy()
processed_val = X_val.copy()
processed_prod = X_prod.copy()

processed_train[TARGET] = y_train.values
processed_val[TARGET] = y_val.values
processed_prod[TARGET] = y_prod.values

processed_train[ID_COL] = train_metadata[ID_COL].values
processed_val[ID_COL] = val_metadata[ID_COL].values
processed_prod[ID_COL] = prod_metadata[ID_COL].values

processed_train[TIME_COL] = train_metadata[TIME_COL].values
processed_val[TIME_COL] = val_metadata[TIME_COL].values
processed_prod[TIME_COL] = prod_metadata[TIME_COL].values

In [52]:
processed_train.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
processed_val.to_csv(PROCESSED_DATA_DIR / "validation.csv", index=False)
processed_prod.to_csv(PROCESSED_DATA_DIR / "production_simulation.csv", index=False)

print("Saved processed datasets.")

Saved processed datasets.


In [54]:
joblib.dump(encoders, ARTIFACTS_DIR / "label_encoders.pkl")

preprocessing_metadata = {
    "target": TARGET,
    "id_col": ID_COL,
    "time_col": TIME_COL,
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "numeric_missing_value": NUMERIC_MISSING_VALUE,
    "dropped_cols": DROP_COLS,
    "train_shape": processed_train.shape,
    "validation_shape": processed_val.shape,
    "production_simulation_shape": processed_prod.shape,
}

with open(ARTIFACTS_DIR / "preprocessing_metadata.json", "w") as f:
    json.dump(preprocessing_metadata, f, indent=2)

print("Saved preprocessing artifacts.")

Saved preprocessing artifacts.


In [55]:
print("Train null values:", processed_train.isna().sum().sum())
print("Validation null values:", processed_val.isna().sum().sum())
print("Production null values:", processed_prod.isna().sum().sum())

print("Train columns:", processed_train.shape[1])
print("Validation columns:", processed_val.shape[1])
print("Production columns:", processed_prod.shape[1])

processed_train.head()

Train null values: 0
Validation null values: 0
Production null values: 0
Train columns: 434
Validation columns: 434
Production columns: 434


,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,...,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,isFraud,TransactionID,TransactionDT
0,68.5,4,13926,-999.0,150.0,1,142.0,1,315.0,87.0,...,4,2,2,2,2,1,1405,0,2987000,86400
1,29.0,4,2755,404.0,150.0,2,102.0,1,325.0,87.0,...,4,2,2,2,2,1,1405,0,2987001,86401
2,59.0,4,4663,490.0,150.0,4,166.0,2,330.0,87.0,...,4,2,2,2,2,1,1405,0,2987002,86469
3,50.0,4,18132,567.0,150.0,2,117.0,2,476.0,87.0,...,4,2,2,2,2,1,1405,0,2987003,86499
4,50.0,1,4497,514.0,150.0,2,102.0,1,420.0,87.0,...,3,1,0,1,1,2,767,0,2987004,86506


In [61]:
from pathlib import Path
base_dir = Path.cwd().resolve().parents[0]
print("Base directory:", base_dir)

Base directory: E:\ML Practice - 2026\agentic-modelops
